"""
DETECCIÓN DE ICTUS CON TRANSFER LEARNING
✔ DenseNet121 preentrenado en ImageNet
✔ Fine-tuning progresivo
✔ Optimizado para maximizar Recall de Stroke
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import precision_score, recall_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau"""

In [3]:
!pip install protobuf==5.26.1 --quiet
!pip install --upgrade --quiet tensorflow keras

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score, confusion_matrix, 
                            ConfusionMatrixDisplay, classification_report, 
                            roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

print("TensorFlow:", tf.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 5.26.1 which is incompatible.
TensorFlow: 2.20.0
TensorFlow: 2.20.0


In [ ]:
# ============================================================
# 1️⃣ CARGA DE DATOS (sin augmentation)
# ============================================================
X = np.load("/kaggle/input/imagenes/X_imagenes_procesadas.npy")
y_raw = np.load("/kaggle/input/imagenes/y_etiquetas.npy")

# Añadir canal si falta
if X.ndim == 3:
    X = X[..., np.newaxis]

# Etiquetas binarias
y = np.array([1 if label != 'Normal' else 0 for label in y_raw])

print(f"Shape: {X.shape}")
print(f"Rango: [{X.min():.3f}, {X.max():.3f}]")
print(f"Normal: {np.sum(y==0)}, Stroke: {np.sum(y==1)}")

# Split estratificado
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))
print(f"Class weights: {class_weights}")

💻 Ejecutando en LOCAL: /home/noe/Documentos/F5/project-ai-data-scientistG2/data/images

❌ ERROR: No se encontraron los archivos de imágenes
   Buscando en: /home/noe/Documentos/F5/project-ai-data-scientistG2/data/images

📋 Archivos necesarios:
   - X_imagenes_procesadas.npy
   - y_etiquetas.npy

💡 Opciones:
   1. Ejecuta este notebook en Kaggle donde están los datos
   2. Descarga los datos de Kaggle y colócalos en: /home/noe/Documentos/F5/project-ai-data-scientistG2/data/images
   3. Genera los datos ejecutando el notebook de preprocesamiento de imágenes


FileNotFoundError: No se encontraron archivos en /home/noe/Documentos/F5/project-ai-data-scientistG2/data/images

In [ ]:
# ============================================================
# 2️⃣ MODELO DENSE MEJORADO CON REGULARIZACIÓN
# ============================================================
model = Sequential([
    Flatten(input_shape=X_train.shape[1:]),
    
    Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.Precision(name='precision'), 
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │    25,690,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,858,049 (98.64 MB)

 Trainable params: 25,856,513 (98.63 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [ ]:
# ============================================================
# 3️⃣ CALLBACKS OPTIMIZADOS
# ============================================================
callbacks = [
    EarlyStopping(
        monitor='val_recall', 
        patience=10, 
        mode='max', 
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_recall', 
        factor=0.5, 
        patience=5, 
        mode='max',
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        os.path.join(output_dir, 'mejor_modelo_dense.keras'),
        monitor='val_recall',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print(f"📁 Modelos se guardarán en: {output_dir}")


In [ ]:
# ============================================================
# 4️⃣ ENTRENAMIENTO
# ============================================================
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - auc: 0.6099 - loss: 2.4741 - precision: 0.5122 - recall: 0.2504
Epoch 1: val_recall improved from None to 0.20000, saving model to /kaggle/working/mejor_modelo_dense.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 115ms/step - auc: 0.6570 - loss: 2.3396 - precision: 0.5605 - recall: 0.3474 - val_auc: 0.8076 - val_loss: 2.1364 - val_precision: 0.6552 - val_recall: 0.2000 - learning_rate: 1.0000e-04
Epoch 2/100
58/63 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.7490 - loss: 2.1048 - precision: 0.6392 - recall: 0.5288
Epoch 2: val_recall improved from 0.20000 to 0.48421, saving model to /kaggle/working/mejor_modelo_dense.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - auc: 0.7571 - loss: 2.0486 - precision: 0.6152 - recall: 0.5868 - val_auc: 0.8284 - val_loss: 1.8954 - val_precision: 0.7302 - val_recall: 0.4842 - learning_rate: 1.0000e-04
Epoch 3/100
59/63 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.8520 - loss: 1.8414 - precision: 0.6958 - recall: 

In [ ]:
# ============================================================
# 5️⃣ EVALUACIÓN CON DIFERENTES THRESHOLDS
# ============================================================
def eval_threshold(model, X, y, threshold, label=""):
    y_pred_proba = model.predict(X, verbose=0)
    y_pred = (y_pred_proba > threshold).astype(int).flatten()
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    return precision, recall

print("\n" + "="*60)
print("🎯 EVALUACIÓN CON DIFERENTES THRESHOLDS")
print("="*60)

y_test_proba = model.predict(X_test, verbose=0)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    precision, recall = eval_threshold(model, X_test, y_test, threshold)
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Threshold {threshold:.2f}: Recall={recall:.1%}, Precision={precision:.1%}, F1={f1:.3f}")
    
    if recall >= 0.80 and f1 > best_f1:  # Priorizar recall >= 80%
        best_f1 = f1
        best_threshold = threshold

print(f"\n🎯 Mejor threshold (F1={best_f1:.3f}): {best_threshold}")



🎯 EVALUACIÓN CON DIFERENTES THRESHOLDS
Threshold 0.30: Recall=98.9%, Precision=43.5%, F1=0.605
Threshold 0.35: Recall=98.9%, Precision=45.0%, F1=0.618
Threshold 0.40: Recall=98.9%, Precision=45.6%, F1=0.625
Threshold 0.45: Recall=98.9%, Precision=47.7%, F1=0.644
Threshold 0.50: Recall=98.9%, Precision=49.2%, F1=0.657
Threshold 0.55: Recall=97.9%, Precision=50.8%, F1=0.669
Threshold 0.60: Recall=97.9%, Precision=54.4%, F1=0.699

🎯 Mejor threshold (F1=0.699): 0.6


In [ ]:
# ============================================================
# 6️⃣ TEST DE ROBUSTEZ (sin rotaciones ni augmentation)
# ============================================================
print("\n" + "="*60)
print("🧪 TEST DE ROBUSTEZ")
print("="*60)

# Solo tests que tienen sentido para TACs
print("\n1️⃣ Robustez a RUIDO (escáneres diferentes):")
for noise_level in [0.01, 0.02, 0.05]:
    X_noisy = X_test + np.random.normal(0, noise_level, X_test.shape)
    X_noisy = np.clip(X_noisy, 0, X_test.max())
    _, recall = eval_threshold(model, X_noisy, y_test, best_threshold)
    print(f"  Ruido σ={noise_level:.2f}: Recall={recall:.1%}")

print("\n2️⃣ Robustez a CONTRASTE (diferentes ventanas HU):")
for contrast in [0.85, 0.95, 1.05, 1.15]:
    X_contrast = np.clip(X_test * contrast, 0, X_test.max())
    _, recall = eval_threshold(model, X_contrast, y_test, best_threshold)
    print(f"  Contraste x{contrast:.2f}: Recall={recall:.1%}")


🧪 TEST DE ROBUSTEZ

1️⃣ Robustez a RUIDO (escáneres diferentes):
  Ruido σ=0.01: Recall=97.9%
  Ruido σ=0.02: Recall=97.9%
  Ruido σ=0.05: Recall=97.9%

2️⃣ Robustez a CONTRASTE (diferentes ventanas HU):
  Contraste x0.85: Recall=98.9%
  Contraste x0.95: Recall=97.9%
  Contraste x1.05: Recall=97.9%
  Contraste x1.15: Recall=97.9%


In [ ]:
# ============================================================
# 7️⃣ EVALUACIÓN FINAL
# ============================================================
print("\n" + "="*60)
print("📊 REPORTE FINAL")
print("="*60)

y_pred_best = (y_test_proba > best_threshold).astype(int).flatten()

print(f"\n🎯 Threshold óptimo: {best_threshold}")
print(classification_report(y_test, y_pred_best, 
                          target_names=['Normal', 'Stroke'],
                          digits=4))

cm = confusion_matrix(y_test, y_pred_best)
print("\n📊 MATRIZ DE CONFUSIÓN:")
print(cm)
print(f"\nTrue Negatives:  {cm[0,0]} (pacientes sanos bien clasificados)")
print(f"False Positives: {cm[0,1]} (falsa alarma)")
print(f"False Negatives: {cm[1,0]} (ictus NO detectado - CRÍTICO)")
print(f"True Positives:  {cm[1,1]} (ictus detectado correctamente)")

if (cm[1,1] + cm[1,0]) > 0:
    recall_final = cm[1,1] / (cm[1,1] + cm[1,0])
    print(f"\n🔴 RECALL STROKE FINAL: {recall_final:.1%}")

auc = roc_auc_score(y_test, y_test_proba)
print(f"📈 AUC-ROC: {auc:.4f}")

# Overfitting
p_train, r_train = eval_threshold(model, X_train, y_train, best_threshold)
p_val, r_val = eval_threshold(model, X_val, y_val, best_threshold)
print(f"\n⚖️  Overfitting:")
print(f"  Train Recall: {r_train:.1%}")
print(f"  Val Recall:   {r_val:.1%}")
print(f"  Test Recall:  {recall_final:.1%}")
print(f"  Diferencia:   {(r_train - recall_final)*100:.2f}%")


📊 REPORTE FINAL

🎯 Threshold óptimo: 0.6
              precision    recall  f1-score   support

      Normal     0.9750    0.5000    0.6610       156
      Stroke     0.5439    0.9789    0.6992        95

    accuracy                         0.6813       251
   macro avg     0.7594    0.7395    0.6801       251
weighted avg     0.8118    0.6813    0.6755       251


📊 MATRIZ DE CONFUSIÓN:
[[78 78]
 [ 2 93]]

True Negatives:  78 (pacientes sanos bien clasificados)
False Positives: 78 (falsa alarma)
False Negatives: 2 (ictus NO detectado - CRÍTICO)
True Positives:  93 (ictus detectado correctamente)

🔴 RECALL STROKE FINAL: 97.9%
📈 AUC-ROC: 0.9126

⚖️  Overfitting:
  Train Recall: 99.5%
  Val Recall:   98.9%
  Test Recall:  97.9%
  Diferencia:   1.58%


In [ ]:
# %%
# ============================================================
# 5️⃣ GUARDADO DEL MODELO EN KAGGLE
# ============================================================
import os
import json

print("💾 Guardando modelo de imágenes en Kaggle...")

# Calcular métricas si no existen
try:
    p_test
except NameError:
    print("⚠️ Calculando métricas primero...")
    
    def eval_model(model, X, y, label):
        y_pred = (model.predict(X, verbose=0) > 0.5).astype(int)
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)
        print(f"{label} - Precision: {precision:.4f}, Recall: {recall:.4f}")
        return precision, recall
    
    p_train, r_train = eval_model(model, X_train, y_train, "Train")
    p_val, r_val     = eval_model(model, X_val, y_val, "Validation")
    p_test, r_test   = eval_model(model, X_test, y_test, "Test")

# output_dir ya está definido al inicio del notebook
print(f"💾 Guardando en: {output_dir}")

# Crear directorio si no existe
os.makedirs(output_dir, exist_ok=True)

# Guardar modelo completo
model_path = os.path.join(output_dir, "stroke_image_model.h5")
model.save(model_path)

print(f"✅ Modelo guardado en: {model_path}")

# Guardar también métricas y metadatos
metadata = {
    'model_type': 'DenseNet_Binary_Classifier',
    'input_shape': list(X_train.shape[1:]),
    'test_precision': float(p_test),
    'test_recall': float(r_test),
    'train_precision': float(p_train),
    'train_recall': float(r_train),
    'val_precision': float(p_val),
    'val_recall': float(r_val),
    'class_weights': {str(k): float(v) for k, v in class_weights.items()},
    'threshold': 0.5,
    'trained_on': 'Kaggle',
    'framework': 'TensorFlow/Keras'
}

metadata_path = os.path.join(output_dir, "stroke_image_model_metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✅ Metadatos guardados en: {metadata_path}")

print("\n📊 Resumen del modelo:")
print(f"   Recall en Test: {r_test:.2%}")
print(f"   Precision en Test: {p_test:.2%}")
print(f"   Shape de entrada: {X_train.shape[1:]}")

print("\n📥 Para descargar:")
print("   1. Ve a la pestaña 'Output' en Kaggle")
print("   2. Descarga 'stroke_image_model.h5'")
print("   3. Descarga 'stroke_image_model_metadata.json'")
print("   4. Colócalos en la carpeta 'models/' de tu proyecto")

💾 Guardando modelo de imágenes en Kaggle...
⚠️ Calculando métricas primero...
Train - Precision: 0.5108, Recall: 0.9987
Validation - Precision: 0.4921, Recall: 0.9895


Test - Precision: 0.4921, Recall: 0.9895
✅ Modelo guardado en: /kaggle/working/stroke_image_model.h5
✅ Metadatos guardados en: /kaggle/working/stroke_image_model_metadata.json

📊 Resumen del modelo:
   Recall en Test: 98.95%
   Precision en Test: 49.21%
   Shape de entrada: (224, 224, 1)

📥 Para descargar:
   1. Ve a la pestaña 'Output' en Kaggle
   2. Descarga 'stroke_image_model.h5'
   3. Descarga 'stroke_image_model_metadata.json'
   4. Colócalos en la carpeta 'models/' de tu proyecto


In [ ]:
# ============================================================
# 8️⃣ VISUALIZACIONES
# ============================================================
fig = plt.figure(figsize=(16, 10))

# 1. Loss
ax1 = plt.subplot(2, 3, 1)
ax1.plot(history.history['loss'], label='Train', linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation', linewidth=2)
ax1.set_title('Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Época')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Recall
ax2 = plt.subplot(2, 3, 2)
ax2.plot(history.history['recall'], label='Train', linewidth=2)
ax2.plot(history.history['val_recall'], label='Validation', linewidth=2)
ax2.axhline(y=0.8, color='r', linestyle='--', label='Target 80%', alpha=0.7)
ax2.set_title('Recall (Stroke)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Época')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. AUC
ax3 = plt.subplot(2, 3, 3)
ax3.plot(history.history['auc'], label='Train', linewidth=2)
ax3.plot(history.history['val_auc'], label='Validation', linewidth=2)
ax3.set_title('AUC', fontsize=12, fontweight='bold')
ax3.set_xlabel('Época')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Matriz de confusión
ax4 = plt.subplot(2, 3, 4)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Stroke'])
disp.plot(ax=ax4, cmap=plt.cm.Blues, colorbar=False)
ax4.set_title(f'Matriz de Confusión (threshold={best_threshold})', 
              fontsize=12, fontweight='bold')

# 5. Curva ROC
ax5 = plt.subplot(2, 3, 5)
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
ax5.plot(fpr, tpr, label=f'Dense Model (AUC={auc:.3f})', linewidth=2)
ax5.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax5.axhline(y=0.8, color='r', linestyle=':', alpha=0.7, label='Target Recall')
ax5.set_xlabel('False Positive Rate')
ax5.set_ylabel('True Positive Rate')
ax5.set_title('Curva ROC', fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Distribución de probabilidades
ax6 = plt.subplot(2, 3, 6)
stroke_probs = y_test_proba[y_test == 1].flatten()
normal_probs = y_test_proba[y_test == 0].flatten()
ax6.hist(normal_probs, bins=30, alpha=0.6, label='Normal', color='blue', density=True)
ax6.hist(stroke_probs, bins=30, alpha=0.6, label='Stroke', color='red', density=True)
ax6.axvline(x=best_threshold, color='green', linestyle='--', 
            linewidth=2, label=f'Threshold={best_threshold}')
ax6.set_xlabel('Probabilidad predicha')
ax6.set_ylabel('Densidad')
ax6.set_title('Distribución de Probabilidades', fontsize=12, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
eval_plot_path = os.path.join(output_dir, 'dense_model_evaluation.png')
plt.savefig(eval_plot_path, dpi=150)
plt.show()

print("\n✅ Modelo Dense optimizado y evaluado")
print(f"📁 Modelo guardado en: {os.path.join(output_dir, 'mejor_modelo_dense.keras')}")
print(f"📊 Gráfico guardado en: {eval_plot_path}")